# Live MagPy magnetometer display

This tutorial starts the MAGIE live dashboard from a local MagPyBin file or an SSH source. The dashboard shows magnetic components, temperatures, location, a compass and a provisional K-index chart.

Use an environment with MAGIE and its dependencies available. The examples use `python -m magie.live_magpy_display`, so the `magie-live` shell command does not need to be installed separately. The notebook's Python environment must be able to import `magie`.

## Quick start

Run one of these commands in a terminal where MAGIE is available. Replace the example paths, filename pattern and station name with your own.

**Local daily files:**

```bash
python -m magie.live_magpy_display --local /path/to/data/ --filename 'sensor_{date}.bin' --station 'Station name'
```

**Daily files over SSH:**

```bash
python -m magie.live_magpy_display --ssh user@host:/absolute/path/to/data/ --filename 'sensor_{date}.bin' --station 'Station name'
```

For SSH, replace `user@host` and the absolute path with your own SSH configuration. SSH key authentication and host verification must work without an interactive prompt. In either command, replace `sensor_` with the actual filename prefix while keeping `{date}` if the files use a `YYYY-MM-DD` date. For a single fixed file, give its full path to `--local` or `--ssh` and omit `--filename`.

Open **http://127.0.0.1:8765/** on the computer running the display and keep the terminal open. Press **Ctrl+C** to stop it. For provisional K, the station name must match a recognised MAGIE station, or you can provide its recognised alias with `--k-site`. The remaining cells explain the options and provide a configurable command.

## Example display

![The MagPy display showing magnetic readings and provisional K values](assets/live_magpy_display_example.png)

This screenshot shows a moment in time; readings and provisional K values continue to change. The source label has been replaced with `Example data` for the tutorial.

In [ ]:
import sys
import magie.live_magpy_display

print("Python:", sys.executable)
print("Display module:", magie.live_magpy_display.__file__)

## Choose a data source

Set `source_type` to `"local"` or `"ssh"` and replace the example path. For SSH, the format is `user@host:/absolute/path/`; configure SSH keys and host verification beforehand, since the display uses non-interactive `scp`. Do not put passwords in this notebook.

For a **daily-file directory**, set `filename` to the instrument's exact pattern. `{date}` is replaced by the current UTC date as `YYYY-MM-DD` and allows the display to follow daily rollover and load earlier files. For a single fixed file, set `filename = None` and use its full path as `source`.

Local directories also accept filename globs such as `"*.bin"`. A directory without `filename` uses the lexicographically last `*.bin` file; to load a rolling three-day history, provide a dated pattern.

In [ ]:
source_type = "local"  # "local" or "ssh"
source = "/path/to/magnetometer/files/"  # or "user@host:/absolute/path/"
filename = "sensor_{date}.bin"  # replace with your pattern, or None for one file
station_name = "LEMI magnetometer"  # displayed on the page
k_site = None  # set a recognised MAGIE station alias, or use a recognised station_name
port = 8765
interval_seconds = 1
preserve_xyz_divisors = False  # True when the file's XYZ header divisors are correct

## Build the command

The next cell only prints the command; it does not start a server or connect to SSH. Use a recognised `k_site` when you want the provisional K-index calculation. Unknown stations need suitable K9 and longitude metadata before K can be calculated.

In [ ]:
import shlex

if source_type not in {"local", "ssh"}:
    raise ValueError('source_type must be "local" or "ssh"')

command = [sys.executable, "-m", "magie.live_magpy_display",
           "--local" if source_type == "local" else "--ssh", source,
           "--station", station_name, "--port", str(port),
           "--interval", str(interval_seconds)]
if filename:
    command += ["--filename", filename]
if k_site:
    command += ["--k-site", k_site]
if preserve_xyz_divisors:
    command += ["--preserve-xyz-divisors"]

print(shlex.join(command))

## Start and stop the display

After replacing the placeholders, either copy the printed command into a terminal, or set `start_dashboard = True` below and run the cell. The process continues in the background until you stop it. If a server is already running on the selected port, stop it or choose another port.

On the same machine as the server, open **http://127.0.0.1:8765/** (replace the port if changed). Keep the process running while using the page. If the notebook is running on a different machine, use your notebook platform's port forwarding or an SSH tunnel to access that machine's localhost.

In [ ]:
import subprocess

start_dashboard = False  # set True after configuring the source above
if start_dashboard:
    if "dashboard_process" in globals() and dashboard_process.poll() is None:
        print("The dashboard started by this notebook is already running.")
    else:
        dashboard_process = subprocess.Popen(command, stdin=subprocess.DEVNULL)
        print(f"Started PID {dashboard_process.pid}. Open http://127.0.0.1:{port}/")
        print("If it exits, check the notebook/terminal logs for the error.")
else:
    print("Not started. Set start_dashboard = True when ready.")

To stop a process started by this notebook, run the following cell. If you started it in a terminal instead, press **Ctrl+C** there.

In [ ]:
if "dashboard_process" in globals() and dashboard_process.poll() is None:
    dashboard_process.terminate()
    try:
        dashboard_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        print("The dashboard has not stopped yet; check the process before continuing.")
    else:
        print("Dashboard stopped.")
else:
    print("No running dashboard process started by this notebook.")

## Use the dashboard

- Select **X, Y, Z, H, D** or a temperature card to change the plot. H is `sqrt(X² + Y²)`; D is `atan2(Y, X)` in degrees. The compass is relative to the input X axis and represents geographic declination only when X and Y are calibrated and rotated to geographic north and east.
- Choose a time range up to **3 days**. Drag a box to zoom, choose **Pan** to drag the window, or use **Reset view** to return to the selected range. Move over the plot to inspect a reading.
- Click the compass, line plot or K chart, or use the expand buttons, for a larger view. **Back to dashboard** restores the previous layout. In the compass view, zero means positive X is aligned with the measured local horizontal field (magnetic north). The green and amber thresholds are adjustable display guides; the colour cue is disabled for stale, disconnected or invalid readings. Four decimal places do not imply instrument accuracy.
- **Full screen** provides a compact layout for a display. Use the button again or press Escape to exit; small or portrait screens can scroll.
- Temperature channels are read from the file; for TE/TF labels the dashboard identifies electronics and sensor temperatures respectively. The gauge graphic is limited to −20–50 °C, but the numeric values are not clipped.
- The map uses the coordinates reported by the instrument. Loading the Google map requires internet access and sends those coordinates to Google.
- Check the data age and error message if updates stop. The monitor targets a one-second refresh, but a full file is copied/read each time and a slow transfer can take longer. `interval_seconds = 30` reduces network and CPU load.

The display corrects the XYZ header divisors to `1` in a temporary copy by default, addressing the incorrect divisors in the LEMI data for which this display was developed. The source file and temperature divisors are untouched. Use `preserve_xyz_divisors = True` for files whose XYZ header divisors are already correct.

## Provisional live K index

The K panel uses MAGIE's operational live FMI approach with station-specific K9 and longitude, future missing-value padding, and filtered LEMI observations. It does not use the experimental anomaly-removal workflow. Values are **provisional**; the current three-hour interval can change.

With a `{date}` template, the worker can load up to four preceding UTC days for K context. It needs at least 45 hours of preceding data, with at least 80% coverage in that history and in the displayed interval. The current interval requires at least 30 minutes of observations, and FMI may still return an unavailable value. Raw instrument axes are not automatically rotated or anomaly-cleaned.

K calculation runs in a separate worker at approximately one-minute intervals. It may take time to load historical files; missing or incomplete data can make values unavailable. History is kept in memory until the process stops. The 3-day plot is also subject to the `--limit` sample limit (259,200 by default), which may need increasing for faster-than-1 Hz data.

## Further options and troubleshooting

Run the following cell to see the full command-line help. `--interval`, `--limit`, `--port`, `--station`, `--k-site`, `--temperature-key`, `--xyz-scale`, and `--preserve-xyz-divisors` are available.

If the page reports an SSH failure, check that the path and daily filename pattern are correct and that `ssh`/`scp` work without an interactive password prompt. If the source is on the same machine as the dashboard, use a local path instead. If `127.0.0.1` cannot be reached, check where the dashboard process is running and whether the port is already in use.

In [ ]:
subprocess.run([sys.executable, "-m", "magie.live_magpy_display", "--help"], check=True);